# EDA — Atlas Precios

Análisis exploratorio del estado actual del proyecto: **4 cadenas** (Coto, Día, Carrefour, Jumbo), índice de la Canasta Atlas total y por categoría, comparador entre cadenas y contexto macro (IPC / dólar).

El notebook se regenera solo a medida que entra data — no hay cifras hardcodeadas. El índice y las series históricas usan la **cadena de referencia** (`coto`) con `precio_lista` y **canasta fija** (productos con serie completa), la misma metodología que el pipeline y el dashboard.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
FUENTE_REF = "coto"

DB = Path("data/atlas.db")
if not DB.exists():
    DB = Path("../data/atlas.db")
con = sqlite3.connect(DB)
df = pd.read_sql(
    """SELECT pr.fecha, pr.fuente, pr.precio_lista, pr.precio_promo,
              p.nombre_normalizado AS producto, p.categoria, p.ean
       FROM precios pr JOIN productos p ON p.id = pr.producto_id
       WHERE p.en_canasta = 1""", con)
reg = pd.read_sql("SELECT fecha, serie, valor FROM regresores", con)
eventos = pd.read_sql("SELECT fecha, tipo, detalle FROM eventos ORDER BY fecha DESC", con)
con.close()

df["fecha"] = pd.to_datetime(df["fecha"])
print(f"{df['fecha'].nunique()} días · {df['fuente'].nunique()} cadenas · {df['producto'].nunique()} productos-fila")
df.head()

## 1. Cobertura de datos

Cuántos productos relevó cada cadena por día. Coto (referencia) tiene la serie más larga; las demás se suman a partir de v2.

In [ ]:
cobertura = df.pivot_table(index="fecha", columns="fuente", values="producto",
                           aggfunc="nunique", fill_value=0)
print(cobertura.to_string())

## 2. Índice Canasta Atlas (total)

Costo de una canasta **fija** (productos con precio todos los días), base 100 al primer día. Solo `precio_lista`, cadena de referencia.

In [ ]:
def costo_canasta_fija(g):
    """Serie fecha -> costo, usando solo productos presentes TODOS los días."""
    piv = g.pivot_table(index="fecha", columns="producto", values="precio_lista").sort_index()
    return piv.dropna(axis=1).sum(axis=1)

ref = df[df["fuente"] == FUENTE_REF]
costo = costo_canasta_fija(ref)
indice = (costo / costo.iloc[0] * 100).round(2)

n_fijos = ref.pivot_table(index="fecha", columns="producto", values="precio_lista").dropna(axis=1).shape[1]
var = indice.iloc[-1] - 100
print(f"Canasta fija: {n_fijos} productos con serie completa")
print(f"Variación {indice.index[0].date()} -> {indice.index[-1].date()}: {var:+.2f}%")
print(f"Costo canasta: ${costo.iloc[0]:,.0f} -> ${costo.iloc[-1]:,.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(indice.index, indice.values, marker="o", color="#1a2744")
ax.axhline(100, color="#c9a227", ls="--", lw=1)
ax.set_title("Índice Canasta Atlas — base 100"); ax.set_ylabel("Índice")
ax.tick_params(axis="x", rotation=45); fig.tight_layout(); plt.show()

## 3. Índice por categoría

El mismo cálculo, desagregado en las 6 categorías. Muestra qué rubros empujan la canasta.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
filas = []
for cat, g in ref.groupby("categoria"):
    c = costo_canasta_fija(g)
    if c.empty:
        continue
    idx_cat = c / c.iloc[0] * 100
    ax.plot(idx_cat.index, idx_cat.values, marker=".", label=cat.capitalize())
    filas.append({"categoria": cat, "var_%": round(idx_cat.iloc[-1] - 100, 2)})
ax.axhline(100, color="#bbb", ls=":"); ax.legend(fontsize=8, ncol=3)
ax.set_title("Índice por categoría — base 100"); ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); plt.show()

pd.DataFrame(filas).sort_values("var_%", ascending=False).reset_index(drop=True)

## 4. Mayores movimientos por producto

Subas y bajas del precio de lista en la cadena de referencia entre el primer y el último día.

In [ ]:
piv = ref.pivot_table(index="producto", columns="fecha", values="precio_lista")
primero, ultimo = piv.columns.min(), piv.columns.max()
mov = ((piv[ultimo] - piv[primero]) / piv[primero] * 100).dropna().sort_values(ascending=False)
print("TOP SUBAS (%)"); print(mov.head(5).round(1).to_string())
print("\nTOP BAJAS (%)"); print(mov.tail(5).round(1).to_string())

## 5. Comparador entre cadenas

Mismo producto (match por EAN), precio de cada cadena. Se usa la fecha con **más cadenas relevadas** y los productos con precio en todas ellas. Los frescos de balanza no cross-matchean, a propósito.

In [ ]:
# Fecha con mayor cobertura de cadenas
cad_por_fecha = df.groupby("fecha")["fuente"].nunique()
fecha_comp = cad_por_fecha[cad_por_fecha == cad_por_fecha.max()].index.max()
d = df[df["fecha"] == fecha_comp]
fuentes = sorted(d["fuente"].unique())

piv_c = d.pivot_table(index="producto", columns="fuente", values="precio_lista")
comp = piv_c.dropna(subset=fuentes)
print(f"Fecha: {fecha_comp.date()} · cadenas: {fuentes} · productos comparables: {len(comp)}")

totales = comp[fuentes].sum()
print("\nCanasta comparable por cadena:")
for f in fuentes:
    print(f"  {f:10s} ${totales[f]:,.0f}")
print(f"  -> más barata: {totales.idxmin()}")
print("\nGanador por producto:", comp[fuentes].idxmin(axis=1).value_counts().to_dict())

ax = comp[fuentes].plot(kind="barh", figsize=(9, max(4, 0.35 * len(comp))))
ax.set_title(f"Precio por cadena — {fecha_comp.date()}"); ax.set_xlabel("Precio ($)")
plt.tight_layout(); plt.show()

## 6. Contexto macro — IPC oficial y dólar

Regresores externos. Cuando la canasta acumule semanas, se podrá contrastar nuestra medición diaria propia contra la inflación oficial del INDEC (mensual, con rezago).

In [ ]:
ipc = reg[reg["serie"] == "ipc"].copy()
if len(ipc) >= 2:
    ipc["fecha"] = pd.to_datetime(ipc["fecha"]); ipc = ipc.sort_values("fecha")
    ipc["var_mensual"] = ipc["valor"].pct_change() * 100
    ult = ipc.iloc[-1]
    interanual = (ult["valor"] / ipc.iloc[-13]["valor"] - 1) * 100 if len(ipc) >= 13 else None
    print(f"IPC INDEC — último mes ({ult['fecha'].strftime('%b %Y')}): {ult['var_mensual']:.1f}%")
    if interanual is not None:
        print(f"IPC interanual: {interanual:.1f}%")
    fig, ax = plt.subplots(figsize=(9, 3.5))
    tail = ipc.tail(12)
    ax.bar(tail["fecha"].dt.strftime("%b %y"), tail["var_mensual"], color="#1a2744")
    ax.set_title("IPC INDEC — variación mensual (últimos 12 meses)"); ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=45); fig.tight_layout(); plt.show()
else:
    print("Aún sin serie de IPC suficiente.")

## 7. Control de calidad

El índice usa `precio_lista` porque `precio_promo` tuvo bugs históricos (valores ×80). Los guards de sanidad garantizan que ningún promo absurdo sobreviva en la base.

In [ ]:
absurdos = df[df["precio_promo"].notna() & (df["precio_promo"] >= df["precio_lista"])]
print(f"Promos absurdos (>= precio_lista) en la base: {len(absurdos)}  (esperado: 0)")
print(f"Registros con promo válido: {df['precio_promo'].notna().sum()}")
print(f"\nEventos registrados: {len(eventos)}")
if len(eventos):
    print(eventos['tipo'].value_counts().to_string())